# Controlling the binomial/Kendall inflation — the LFMM way (site unit + latent factors)

The plot-level raw binomial/Kendall (bio1, gen9) are wildly inflated (binomial GIF≈105, Kendall GIF≈9) from two sources: **pseudoreplication** (355 pools carry only 31 independent climate values) and, for the binomial, **count over-precision** (AF×flowers×2 treated as independent draws). Genomic control alone flattened everything to 0 hits. Here are the two principled fixes, each the analog of an LFMM ingredient:

- **(A) Honest 31-site unit** — flower-weighted site-mean AF; one obs per climate value. Removes pseudoreplication at the source (Kendall becomes genuinely calibrated; binomial loses the 355→31 inflation but keeps some count over-precision → still GIF-check it).
- **(B) Latent-factor binomial (K=16)** — top-16 pool structure factors (PCs of the class AF matrix = LFMM's `U`) added as GLM covariates, so the climate slope is tested net of structure. This is LFMM's engine dropped into the binomial. **Kendall takes no covariates** (rank correlation, no design matrix), so (B) is binomial-only.

In [ ]:
import sys, os
PROJ='/global/scratch/users/tbellg/kmate'
sys.path.insert(0, f'{PROJ}/analysis/grenenet_gea')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
import scipy.stats as st, lib
from scipy.stats import chi2
mpl.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
GNP=f'{PROJ}/analysis/grenenet_gea/gea_newpanel/results'
OUT=f'{GNP}/binom_kendall_controlled'; os.makedirs(OUT, exist_ok=True)
CHR=[f'Chr{i}' for i in range(1,6)]
ACC={'snp':'#c1443c','nonsnp':'#2e7d5b'}; NAME={'snp':'SNP','nonsnp':'non-SNP (indel+SV)'}
NULLMED=chi2.ppf(0.5,1)   # 0.4549
cols={'Chr1':'#3b5b92','Chr2':'#8bb0d0','Chr3':'#3b5b92','Chr4':'#8bb0d0','Chr5':'#3b5b92'}
def _bh(p):
    p=np.asarray(p,float); n=len(p); o=np.argsort(p); q=np.empty(n)
    q[o]=(p[o]*n)/(np.arange(n)+1); q[o]=np.minimum.accumulate(q[o][::-1])[::-1]; return np.clip(q,0,1)
def gif_of(p):
    p=np.asarray(p,float); p=p[np.isfinite(p)].clip(1e-300,1.0)
    return float(np.median(chi2.isf(p,1))/NULLMED)
genes=lib.load_genes()
def annotate(t):
    a=lib.annotate_svs(t.copy(), flank=2000, genes=genes)
    t=t.copy(); t['gene_name']=a.gene_name.values; t['genes']=a.genes_all.values; return t

Genome-coordinate + loader helpers (shared by both sections).

In [ ]:
def add_gx(D, off):
    D['gx']=D.pos.astype(float)+D.chrom.map(off); return D
def offsets(frames):
    chrlen={c:int(max(F.loc[F.chrom==c,'pos'].max() for F in frames)) for c in CHR}
    off={}; run=0
    for c in CHR: off[c]=run; run+=chrlen[c]+int(2e6)
    ticks=[off[c]+chrlen[c]/2 for c in CHR]; return off, ticks
def manhattan(frames_by_cls, title, fname, off, ticks, pcol='pval', mafcol=None, maf=0.05):
    fig,axes=plt.subplots(2,1,figsize=(13,6.5),sharex=True)
    for ax,cls in zip(axes,['snp','nonsnp']):
        d=frames_by_cls[cls].copy()
        if mafcol is not None: d=d[d[mafcol]>=maf]
        d=d[np.isfinite(d[pcol])].copy(); d['_nlp']=-np.log10(d[pcol].clip(1e-300))
        pv=d[pcol].to_numpy(float); nt=len(pv); bonf=-np.log10(0.05/nt)
        pas=pv[_bh(pv)<0.05]; fdr=float(-np.log10(pas.max())) if pas.size else None
        base=d.sample(frac=min(1.0,250000/len(d)),random_state=0) if len(d)>250000 else d
        t=pd.concat([base,d[d._nlp>2]]).drop_duplicates(subset=['gx','_nlp'])
        ax.scatter(t.gx,t._nlp,s=3,c=t.chrom.map(cols),rasterized=True,linewidths=0)
        ax.axhline(bonf,ls='--',lw=1.0,color='#444',label=f'Bonferroni ({bonf:.1f})')
        if fdr is not None: ax.axhline(fdr,ls=':',lw=1.3,color='#b8860b',label=f'FDR q<.05 ({fdr:.1f})')
        else: ax.plot([],[],' ',label='FDR q<.05: none pass')
        top=d.nlargest(1,'_nlp').iloc[0]
        ax.scatter([top.gx],[top._nlp],s=42,facecolors='none',edgecolors=ACC[cls],linewidths=1.6,zorder=5)
        ax.annotate(f'{top.chrom}:{int(top.pos):,} {top.block}',(top.gx,top._nlp),
                    xytext=(6,-1),textcoords='offset points',fontsize=8,color=ACC[cls])
        gif=gif_of(pv)
        ax.set_ylabel('-log10 p'); ax.text(0.995,0.90,f'{NAME[cls]}  (GIF={gif:.1f}, n={nt:,})',
                    transform=ax.transAxes,ha='right',fontweight='bold',color=ACC[cls])
        ax.set_ylim(-0.4, max(float(d._nlp.max()),bonf)*1.10)
        ax.legend(loc='upper left',fontsize=7.5,frameon=False,handlelength=1.6)
    axes[1].set_xticks(ticks); axes[1].set_xticklabels(CHR); axes[1].set_xlabel('genome position')
    fig.suptitle(title,y=0.98); fig.tight_layout()
    fig.savefig(f'{OUT}/{fname}',dpi=150,bbox_inches='tight'); plt.show(); print('wrote',fname)

## (A) Honest 31-site unit — binomial + Kendall
Flower-weighted site-mean AF, one observation per climate value (31 sites). `site_maf≥0.05`. This removes pseudoreplication; compare GIF here to the plot-level ≈105/9.

In [ ]:
site={}
for test in ['binomial','kendall']:
    for cls in ['snp','nonsnp']:
        d=pd.read_csv(f'{GNP}/{test}_site/{test}_{cls}_site_bio1.csv')
        d=d[np.isfinite(d.pval)&(d.site_maf>=0.05)].copy()
        site[(test,cls)]=d
offA,ticksA=offsets(list(site.values()))
for d in site.values(): add_gx(d,offA)
print(f'{"scan (site)":18s} {"n":>9s} {"GIF":>6s} {"min p":>9s} {"p<1e-3":>7s} {"FDR<.05":>8s} {"Bonf":>5s}')
for test in ['binomial','kendall']:
    for cls in ['snp','nonsnp']:
        d=site[(test,cls)]; p=d.pval.to_numpy(float); n=len(p)
        print(f'{test+"/"+cls:18s} {n:>9,} {gif_of(p):6.1f} {p.min():9.1e} '
              f'{int((p<1e-3).sum()):>7,} {int((_bh(p)<0.05).sum()):>8,} {int((p<0.05/n).sum()):>5,}')

### Site-level Manhattans (binomial, then Kendall)

In [ ]:
manhattan({c:site[('binomial',c)] for c in ['snp','nonsnp']},
          'Site-level (31-unit) binomial — bio1', 'manhattan_site_binomial.png', offA, ticksA)
manhattan({c:site[('kendall',c)] for c in ['snp','nonsnp']},
          'Site-level (31-unit) Kendall — bio1', 'manhattan_site_kendall.png', offA, ticksA)

### Site-level top hits (annotated) — Kendall (the calibrated test)

In [ ]:
def site_top(test,cls,n=15):
    stat='slope' if test=='binomial' else 'tau'
    t=site[(test,cls)].nsmallest(n,'pval')[['chrom','pos','ref_len','alt_len','site_maf','block',stat,'pval']]
    return annotate(t).reset_index(drop=True)
for test in ['binomial','kendall']:
    for cls in ['snp','nonsnp']:
        site_top(test,cls).to_csv(f'{OUT}/site_top_{test}_{cls}.csv',index=False)
pd.set_option('display.width',220,'display.max_colwidth',34)
site_top('kendall','nonsnp')

Kendall site-level — top SNP records

In [ ]:
site_top('kendall','snp')

## (B) Latent-factor binomial (K=16) — LFMM's structure covariate inside the binomial
Plot-level binomial GLM with the top-16 pool structure factors (PCs of the class AF matrix, = LFMM's `U`) added as covariates: `[alt,ref] ~ const + z(bio1) + LF1..LF16`. Compare GIF and hits to the **raw** plot-level binomial (no factors). If latent factors are doing LFMM's job, the GIF should fall sharply toward 1.

In [ ]:
lf={}; raw={}
for cls in ['snp','nonsnp']:
    lf[cls]=pd.read_csv(f'{GNP}/binomial_latent/binomial_lf16_{cls}_gen9_bio1.csv')
    lf[cls]=lf[cls][np.isfinite(lf[cls].pval)].copy()
    raw[cls]=pd.read_csv(f'{GNP}/binomial/binomial_{cls}_gen9_bio1.csv')
    raw[cls]=raw[cls][np.isfinite(raw[cls].pval)].copy()
offB,ticksB=offsets(list(lf.values()))
for d in list(lf.values())+list(raw.values()): add_gx(d,offB)
print(f'{"binomial (plot)":22s} {"GIF":>7s} {"p<1e-5":>9s} {"FDR<.05":>9s} {"Bonf":>6s}')
for cls in ['snp','nonsnp']:
    for tag,d in [('raw (no factors)',raw[cls]),('+K16 latent factors',lf[cls])]:
        p=d.pval.to_numpy(float); n=len(p)
        print(f'{cls+": "+tag:22s} {gif_of(p):7.1f} {int((p<1e-5).sum()):>9,} '
              f'{int((_bh(p)<0.05).sum()):>9,} {int((p<0.05/n).sum()):>6,}')

### QQ — raw binomial vs +K16 latent factors
Grey = raw; coloured = with latent factors. If factors absorb structure, the coloured line drops toward the diagonal.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
for ax,cls in zip(axes,['snp','nonsnp']):
    for d,color,label in [(raw[cls],'#b0b0b0','raw'),(lf[cls],ACC[cls],'+K16 latent factors')]:
        p=np.sort(d.pval.to_numpy(float)); n=len(p)
        exp=-np.log10((np.arange(1,n+1)-0.5)/n); obs=-np.log10(np.clip(p,1e-300,1))
        idx=np.unique(np.r_[np.linspace(0,n-1,2000).astype(int),np.arange(max(0,n-500),n)])
        ax.plot(exp[idx],obs[idx],'.',ms=3,color=color,label=f'{label} (GIF={gif_of(p):.1f})')
    mx=float(max(exp.max(),obs.max())); ax.plot([0,mx],[0,mx],'k--',lw=0.8)
    ax.set_title(f'binomial — {NAME[cls]}',fontsize=10)
    ax.set_xlabel('expected -log10 p'); ax.set_ylabel('observed -log10 p'); ax.legend(fontsize=8,frameon=False)
fig.suptitle('Latent factors in the binomial (LFMM U) — raw vs K=16',y=1.0); fig.tight_layout()
fig.savefig(f'{OUT}/qq_binomial_latent.png',dpi=140,bbox_inches='tight'); plt.show(); print('wrote qq_binomial_latent.png')

### Latent-factor binomial Manhattan + top hits

In [ ]:
manhattan(lf,'Latent-factor (K=16) binomial — bio1, SNP vs non-SNP','manhattan_binomial_latent.png',offB,ticksB)
def lf_top(cls,n=15):
    t=lf[cls].nsmallest(n,'pval')[['chrom','pos','ref_len','alt_len','MAF','block','slope','pval']]
    return annotate(t).reset_index(drop=True)
for cls in ['snp','nonsnp']: lf_top(cls).to_csv(f'{OUT}/latent_top_binomial_{cls}.csv',index=False)
lf_top('nonsnp')

## (C) Do the controls recover the LFMM non-SNP hit?
The LFMM site-PC1 panel's real non-SNP Bonferroni hit was **`Chr1_9323` @ Chr1:13,979,249** (2-bp indel). Check its p under each controlled binomial/Kendall scan — a genuine signal should survive the honest unit and stay non-significant in SNPs.

In [ ]:
def look(frame, chrom='Chr1', pos=13979249, w=200):
    s=frame[(frame.chrom==chrom)&(frame.pos.between(pos-w,pos+w))]
    return s.pval.min() if len(s) else np.nan
print('LFMM non-SNP lead Chr1:13,979,249 (block Chr1_9323) — min p in a +/-200bp window:')
print(f"  site   binomial : nonsnp={look(site[('binomial','nonsnp')]):.2e}  snp={look(site[('binomial','snp')]):.2e}")
print(f"  site   kendall  : nonsnp={look(site[('kendall','nonsnp')]):.2e}  snp={look(site[('kendall','snp')]):.2e}")
print(f"  latent binomial : nonsnp={look(lf['nonsnp']):.2e}  snp={look(lf['snp']):.2e}")

## Read-out
- **(A) Site unit** is the real pseudoreplication fix: Kendall on 31 sites is nonparametric and genuinely calibrated (GIF near 1); binomial on 31 sites drops the 355→31 inflation but keeps residual count over-precision (its site-level GIF, printed above, says how much). The site-level tables are the honest SNP-vs-non-SNP comparison, directly parallel to LFMM site-PC1.
- **(B) Latent factors** are LFMM's actual engine, and they slot into the binomial: the GIF before/after quantifies how much of the inflation was structure the K=16 factors absorb. **Kendall cannot take covariates** — for a structure-corrected rank test you'd move to the site unit or partial correlation; a linear model of Δp with latent factors *is* LFMM.
- **(C)** shows whether the controls preserve the LFMM non-SNP candidate. Convergence of site-level + latent-factor + LFMM on the same non-SNP loci (and their absence in SNPs) is the bar for 'signal SNPs miss' before any selection claim.